In [2]:
#!/usr/bin/env python3
"""
====================================================================
Synthetic pediatric growth cohort generator (males, 0-60 months)
====================================================================

Generates longitudinal weight / length / head-circumference trajectories
with an EXPLICIT, LITERATURE-GROUNDED, AGE-DEPENDENT correlation structure,
for use in the coupled Neural ODE experiments.

Design rationale
----------------
Two dependence structures are imposed SEPARATELY, because the experiments
must be able to distinguish them:

  1. WITHIN-variable tracking (across age).  A child near the 20th weight
     percentile at 6 months tends to stay near it at 12 months.  This is
     what makes FORECASTING possible.  Implemented as a permanent child
     effect plus an Ornstein-Uhlenbeck (mean-reverting) component, so
     tracking decays with age gap toward a floor rather than being
     deterministic.

  2. BETWEEN-variable coupling (at a given age), R(t).  This is what makes
     CROSS-VARIABLE INFERENCE possible.  Because an unobserved variable in
     that task is never measured at any visit, tracking cannot substitute
     for it -- so cross-variable results measure coupling alone.

R(t) is AGE-DEPENDENT and PAIR-SPECIFIC, following Scheffler, Greil &
Hermanussen (2017), Pediatr Res 81:825-830, who report (PCA, n=7,444 boys
and 7,375 girls, 0-7y) that skeletal-growth indicators separate from
fat-deposition indicators, and that AFTER AGE 2 YEARS head volume loads on
a separate third component, indicating independence of head growth.
Accordingly the head-circumference correlations decline across a
transition centred at 24 months, while weight-length is held at a moderate
value throughout.

*** THE COUPLING IS IMPOSED, NOT DISCOVERED. ***
Synthetic data cannot establish that growth variables are biologically
coupled.  It can establish whether a model LEARNS a coupling that is
present.  All claims must be confined to the estimator, not the biology.
The KAPPA sweep (see below) exists so that conclusions do not depend on
committing to one contested correlation value.

Outputs
-------
  cohort_kappa1.00.xlsx   main cohort (kappa = 1)
  cohort_kappa*.csv       sensitivity sweep incl. kappa = 0 negative control
  validation_report.txt   achieved vs target correlations, zone composition

Reproducibility: fully determined by SEED and the CONFIG block below.

Usage:  python generate_cohort.py [--outdir DIR]
"""

import argparse
import os
import sys

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.stats import norm

# =====================================================================
# CONFIG  -- everything reproducible is set here
# =====================================================================

SEED = 20260805
N_CHILDREN = 1020
SEX = 1  # males only

# Clinical visit schedule (months). NOTE: includes 30 months, which was
# absent from the earlier cohort but is required by the paper's protocol
# (historical window = 0-24; forecast horizon = 30, 36, 48, 60).
VISIT_AGES = [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 30.0, 36.0, 48.0, 60.0]
HISTORICAL_CUTOFF = 24.0

# ---------------------------------------------------------------------
# Target between-variable correlations of z-scores.
#
# *** PLACEHOLDER VALUES -- REPLACE WITH VERIFIED LITERATURE ESTIMATES ***
# The SHAPE (head circumference decoupling after 24 months) is grounded in
# Scheffler et al. (2017).  The MAGNITUDES below are plausible defaults and
# must be confirmed against reported correlation coefficients before
# publication.  Experiment 6 (kappa sweep) exists precisely because these
# magnitudes are uncertain.
# ---------------------------------------------------------------------
RHO_WL_EARLY, RHO_WL_LATE = 0.55, 0.50   # weight-length: moderate, ~stable
RHO_WC_EARLY, RHO_WC_LATE = 0.45, 0.15   # weight-headcirc: declines after 24mo
RHO_LC_EARLY, RHO_LC_LATE = 0.45, 0.15   # length-headcirc: declines after 24mo

TRANSITION_CENTER = 24.0  # months; Scheffler's "after age 2 years"
TRANSITION_WIDTH = 6.0    # months; smoothness of the logistic transition

# ---------------------------------------------------------------------
# Within-variable tracking.
#   z(t) = sqrt(lambda)*u  +  sqrt(1-lambda)*OU(t)
# lambda = permanent share (sets the long-run tracking floor)
# tau    = OU correlation time in months
# ---------------------------------------------------------------------
LAMBDA_PERM = {"W": 0.45, "L": 0.55, "C": 0.50}
TAU_OU = {"W": 30.0, "L": 40.0, "C": 36.0}

# ---------------------------------------------------------------------
# Measurement noise (physical units), applied to OBSERVED records only.
# Reflects instrument + operator error.
# ---------------------------------------------------------------------
NOISE_SD = {"W": 0.05, "L": 0.30, "C": 0.20}  # kg, cm, cm

# Missing-visit rate (visits deleted at random to emulate missed clinic
# appointments). Birth visit and forecast-horizon visits are never deleted.
MISSING_RATE = 0.08

# Coupling-strength sweep for Experiment 6. kappa scales the OFF-DIAGONAL
# entries of R(t). kappa = 0 is the negative control (no coupling at all);
# kappa = 1 is the literature-anchored structure.
KAPPA_SWEEP = [0.0, 0.25, 0.5, 0.75, 1.0]
KAPPA_MAIN = 1.0

# ---------------------------------------------------------------------
# Reference distribution: LMS-style parameters per variable per age.
#
# *** REPLACE WITH OFFICIAL WHO MGRS LMS TABLES BEFORE PUBLICATION ***
# M (median) and S (coefficient of variation) below were derived from the
# previously supplied MGRS-calibrated cohort so that the new cohort is on
# a comparable physical scale.  L is the Box-Cox power: L=1 gives a normal
# distribution; L<1 gives right skew (used for weight).
#
# CRITICAL: risk-zone classification uses THESE SAME parameters, so the
# cohort and the classifier are internally consistent by construction.
# Swapping in true WHO LMS changes both together.
# ---------------------------------------------------------------------
REF_AGES = [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 36.0, 48.0, 60.0]

REF_M = {  # median
    "W": [3.490, 5.780, 6.680, 7.655, 8.575, 9.390, 10.570, 11.500, 12.980, 14.695, 16.440],
    "L": [50.915, 58.375, 62.915, 68.440, 72.940, 76.350, 81.885, 85.940, 92.350, 99.120, 105.375],
    "C": [34.920, 39.430, 41.930, 44.445, 45.940, 47.420, 48.950, 49.900, 51.385, 52.225, 52.790],
}
REF_SD = {  # standard deviation in physical units
    "W": [0.491, 0.708, 0.702, 0.689, 0.628, 0.617, 0.629, 0.618, 0.624, 0.790, 0.984],
    "L": [2.279, 2.301, 2.295, 2.292, 2.286, 2.290, 2.298, 2.324, 2.320, 2.615, 2.907],
    "C": [1.522, 1.514, 1.521, 1.521, 1.529, 1.525, 1.518, 1.514, 1.519, 1.528, 1.565],
}
REF_L = {"W": -0.30, "L": 1.0, "C": 1.0}  # Box-Cox power (weight right-skewed)

# Risk-zone band boundaries (percentiles). Matches the paper's 15/85
# Healthy convention; change here to switch to 10/90.
P_CRIT_LO, P_WARN_LO, P_WARN_HI, P_CRIT_HI = 3, 15, 85, 97

VARS = ["W", "L", "C"]
VARNAME = {"W": "Weight_kg", "L": "Length_cm", "C": "HeadCircumference_cm"}


# =====================================================================
# Correlation structure
# =====================================================================

def _transition(ages):
    """Logistic 0->1 ramp centred at TRANSITION_CENTER (0 = early, 1 = late)."""
    return 1.0 / (1.0 + np.exp(-(np.asarray(ages, float) - TRANSITION_CENTER) / TRANSITION_WIDTH))


def target_correlations(ages):
    """Age-dependent pair correlations. Returns dict of arrays over ages."""
    s = _transition(ages)
    return {
        "WL": RHO_WL_EARLY + (RHO_WL_LATE - RHO_WL_EARLY) * s,
        "WC": RHO_WC_EARLY + (RHO_WC_LATE - RHO_WC_EARLY) * s,
        "LC": RHO_LC_EARLY + (RHO_LC_LATE - RHO_LC_EARLY) * s,
    }


def correlation_matrix(rho_wl, rho_wc, rho_lc, kappa=1.0):
    """3x3 correlation matrix with off-diagonals scaled by kappa."""
    R = np.array([
        [1.0, kappa * rho_wl, kappa * rho_wc],
        [kappa * rho_wl, 1.0, kappa * rho_lc],
        [kappa * rho_wc, kappa * rho_lc, 1.0],
    ])
    return _nearest_pd(R)


def _nearest_pd(R):
    """Project to nearest positive-definite correlation matrix (eigenvalue clip)."""
    w, V = np.linalg.eigh(R)
    if w.min() > 1e-8:
        return R
    w = np.clip(w, 1e-8, None)
    R = V @ np.diag(w) @ V.T
    d = np.sqrt(np.diag(R))
    return R / np.outer(d, d)


# =====================================================================
# Latent z-score generation
# =====================================================================

def generate_latent_z(rng, n, ages, kappa):
    """
    Latent standardized growth positions, shape (n, n_ages, 3).

    Step 1: independent per-variable paths with tracking
            z_v(t) = sqrt(lam)*u_v + sqrt(1-lam)*OU_v(t)
    Step 2: at EACH age, mix across variables by the Cholesky factor of
            R(t). Since each variable's path is marginally N(0,1), the
            mixture is marginally N(0,1) and has cross-sectional
            correlation exactly R(t), while tracking is preserved because
            all three variables share the same temporal structure.
    """
    ages = np.asarray(ages, float)
    n_ages = len(ages)
    indep = np.empty((n, n_ages, 3))

    for k, v in enumerate(VARS):
        lam = LAMBDA_PERM[v]
        tau = TAU_OU[v]
        u = rng.standard_normal(n)                      # permanent child effect
        ou = np.empty((n, n_ages))
        ou[:, 0] = rng.standard_normal(n)               # stationary start
        for i in range(1, n_ages):
            dt = ages[i] - ages[i - 1]
            phi = np.exp(-dt / tau)                     # OU autocorrelation
            ou[:, i] = phi * ou[:, i - 1] + np.sqrt(1 - phi**2) * rng.standard_normal(n)
        indep[:, :, k] = np.sqrt(lam) * u[:, None] + np.sqrt(1 - lam) * ou

    rho = target_correlations(ages)
    z = np.empty_like(indep)
    for i in range(n_ages):
        R = correlation_matrix(rho["WL"][i], rho["WC"][i], rho["LC"][i], kappa)
        Lc = np.linalg.cholesky(R)
        z[:, i, :] = indep[:, i, :] @ Lc.T
    return z


# =====================================================================
# z-score  <->  physical units  (LMS)
# =====================================================================

def _ref_interp(var, ages):
    """Interpolate reference M and SD to arbitrary ages (monotone PCHIP)."""
    M = PchipInterpolator(REF_AGES, REF_M[var])(ages)
    SD = PchipInterpolator(REF_AGES, REF_SD[var])(ages)
    return M, SD


def z_to_value(var, z, ages):
    """LMS inverse: z-score -> physical measurement."""
    M, SD = _ref_interp(var, ages)
    S = SD / M                      # coefficient of variation
    Lp = REF_L[var]
    if abs(Lp - 1.0) < 1e-12:
        return M + SD * z
    return M * np.power(np.clip(1.0 + Lp * S * z, 1e-6, None), 1.0 / Lp)


def value_to_z(var, x, ages):
    """LMS forward: physical measurement -> z-score."""
    M, SD = _ref_interp(var, ages)
    S = SD / M
    Lp = REF_L[var]
    if abs(Lp - 1.0) < 1e-12:
        return (x - M) / SD
    return (np.power(x / M, Lp) - 1.0) / (Lp * S)


def classify(z):
    """Risk zone from z-score, using the band boundaries in CONFIG."""
    z_crit_lo, z_warn_lo = norm.ppf(P_CRIT_LO / 100), norm.ppf(P_WARN_LO / 100)
    z_warn_hi, z_crit_hi = norm.ppf(P_WARN_HI / 100), norm.ppf(P_CRIT_HI / 100)
    out = np.full(np.shape(z), "Healthy", dtype=object)
    out[(z >= z_crit_lo) & (z < z_warn_lo)] = "Warning"
    out[(z > z_warn_hi) & (z <= z_crit_hi)] = "Warning"
    out[(z < z_crit_lo) | (z > z_crit_hi)] = "Critical"
    return out


SEVERITY = {"Healthy": 0, "Warning": 1, "Critical": 2}


# =====================================================================
# Cohort assembly
# =====================================================================

def build_cohort(kappa, seed=SEED):
    """Build one cohort at a given coupling strength kappa."""
    rng = np.random.default_rng(seed)
    ages = np.array(VISIT_AGES, float)
    n_ages = len(ages)

    z = generate_latent_z(rng, N_CHILDREN, ages, kappa)

    rows = {
        "Child_ID": np.repeat(np.arange(1, N_CHILDREN + 1), n_ages),
        "Sex": SEX,
        "Age_Months": np.tile(ages, N_CHILDREN),
    }

    age_flat = rows["Age_Months"]
    for k, v in enumerate(VARS):
        z_flat = z[:, :, k].ravel()
        true_val = z_to_value(v, z_flat, age_flat)

        # Observed = true + independent measurement noise
        obs_val = true_val + rng.normal(0.0, NOISE_SD[v], size=true_val.shape)
        obs_z = value_to_z(v, obs_val, age_flat)

        rows[f"{VARNAME[v]}_true"] = np.round(true_val, 3)
        rows[VARNAME[v]] = np.round(obs_val, 2)
        rows[f"z_{v}"] = np.round(z_flat, 4)             # ground-truth z
        rows[f"z_{v}_obs"] = np.round(obs_z, 4)
        rows[f"zone_{v}"] = classify(z_flat)

    df = pd.DataFrame(rows)

    # Overall status = most severe zone across the three variables
    sev = np.stack([df[f"zone_{v}"].map(SEVERITY).values for v in VARS])
    inv = {v: k for k, v in SEVERITY.items()}
    df["zone_overall"] = [inv[s] for s in sev.max(axis=0)]

    # Missingness: emulate missed appointments. Never delete the birth visit
    # (needed as an anchor) nor forecast-horizon visits (needed as ground truth).
    eligible = (df.Age_Months > 0) & (df.Age_Months <= HISTORICAL_CUTOFF)
    df["Observed"] = True
    drop = eligible & (rng.random(len(df)) < MISSING_RATE)
    df.loc[drop, "Observed"] = False
    for v in VARS:
        df.loc[drop, VARNAME[v]] = np.nan
        df.loc[drop, f"z_{v}_obs"] = np.nan

    df["Phase"] = np.where(df.Age_Months <= HISTORICAL_CUTOFF, "Historical", "Forecast")
    df["kappa"] = kappa
    return df


# =====================================================================
# Validation
# =====================================================================

def validate(df, kappa):
    """Check achieved correlations and tracking against targets."""
    lines = []
    ages = np.array(VISIT_AGES, float)
    tgt = target_correlations(ages)

    lines.append(f"--- kappa = {kappa:.2f} ---")
    lines.append("Between-variable correlation of TRUE z-scores (achieved vs target):")
    lines.append(f"{'age':>6} {'r(W,L)':>16} {'r(W,C)':>16} {'r(L,C)':>16}")
    for i, a in enumerate(ages):
        g = df[df.Age_Months == a]
        rwl = g.z_W.corr(g.z_L)
        rwc = g.z_W.corr(g.z_C)
        rlc = g.z_L.corr(g.z_C)
        lines.append(
            f"{a:6.1f} {rwl:7.3f} /{kappa*tgt['WL'][i]:6.3f} "
            f"{rwc:7.3f} /{kappa*tgt['WC'][i]:6.3f} "
            f"{rlc:7.3f} /{kappa*tgt['LC'][i]:6.3f}"
        )

    lines.append("")
    lines.append("Within-variable tracking, corr(z at birth, z at age):")
    for v in VARS:
        p = df.pivot(index="Child_ID", columns="Age_Months", values=f"z_{v}")
        lines.append(f"  {v}: " + " ".join(f"{p[a].corr(p[0.0]):.2f}" for a in ages))

    lines.append("")
    lines.append("Risk-zone composition (all child-visits):")
    for v in VARS:
        vc = df[f"zone_{v}"].value_counts(normalize=True)
        lines.append(
            f"  {v}: H={vc.get('Healthy',0):.1%}  "
            f"W={vc.get('Warning',0):.1%}  C={vc.get('Critical',0):.1%}"
        )
    vc = df.zone_overall.value_counts(normalize=True)
    lines.append(
        f"  overall: H={vc.get('Healthy',0):.1%}  "
        f"W={vc.get('Warning',0):.1%}  C={vc.get('Critical',0):.1%}"
    )
    lines.append("")
    lines.append(f"Missing observed records: {(~df.Observed).mean():.1%}")
    lines.append("")
    return "\n".join(lines)


# =====================================================================
# Main
# =====================================================================

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--outdir", default="./outputs")
    # Parse known args only, ignore extras like -f from Jupyter/Colab
    args, unknown = ap.parse_known_args()

    # If there were unknown arguments, print a warning but continue
    if unknown:
        print(f"Warning: Ignoring unknown arguments: {unknown}", file=sys.stderr)

    os.makedirs(args.outdir, exist_ok=True)

    report = [
        "Synthetic pediatric growth cohort -- validation report",
        "=" * 60,
        f"seed={SEED}  N={N_CHILDREN}  visits={len(VISIT_AGES)}",
        f"visit ages (months): {VISIT_AGES}",
        "",
        "Correlation structure follows Scheffler, Greil & Hermanussen (2017):",
        "head circumference decouples from weight/length after ~24 months.",
        "MAGNITUDES ARE PLACEHOLDERS pending verified literature values.",
        "",
    ]

    for kappa in KAPPA_SWEEP:
        print(f"Generating cohort for kappa = {kappa:.2f}...")
        df = build_cohort(kappa)
        report.append(validate(df, kappa))
        path = os.path.join(args.outdir, f"cohort_kappa{kappa:.2f}.csv")
        df.to_csv(path, index=False)
        print(f"  Saved to: {path}")

        if abs(kappa - KAPPA_MAIN) < 1e-12:
            xls = os.path.join(args.outdir, f"cohort_kappa{kappa:.2f}.xlsx")
            with pd.ExcelWriter(xls, engine="openpyxl") as w:
                df.to_excel(w, sheet_name="cohort", index=False)
                _readme_sheet().to_excel(w, sheet_name="README", index=False)
                _corr_sheet().to_excel(w, sheet_name="R_of_t", index=False)
            print(f"  Saved Excel to: {xls}")

    rpt = os.path.join(args.outdir, "validation_report.txt")
    with open(rpt, "w") as f:
        f.write("\n".join(report))
    print(f"\nValidation report saved to: {rpt}")
    print("\n" + "\n".join(report))


def _readme_sheet():
    return pd.DataFrame({
        "Field": [
            "Child_ID", "Sex", "Age_Months", "Phase", "Observed",
            "Weight_kg / Length_cm / HeadCircumference_cm",
            "*_true", "z_W / z_L / z_C", "z_*_obs",
            "zone_W / zone_L / zone_C", "zone_overall", "kappa",
        ],
        "Description": [
            "Child identifier (1..N)",
            "1 = male (cohort is males only)",
            "Age at visit in months",
            "Historical (<=24mo, conditioning) or Forecast (>24mo)",
            "FALSE = visit missed; measurements are blank",
            "OBSERVED measurement = true value + measurement noise",
            "Noiseless ground-truth value (evaluation only; not model input)",
            "Ground-truth z-score against the reference distribution",
            "z-score of the noisy observed value",
            "Per-variable risk zone from ground-truth z",
            "Most severe zone across the three variables",
            "Coupling-strength scaling of R(t) off-diagonals",
        ],
    })


def _corr_sheet():
    ages = np.array(VISIT_AGES, float)
    t = target_correlations(ages)
    return pd.DataFrame({
        "Age_Months": ages,
        "rho_WL": np.round(t["WL"], 3),
        "rho_WC": np.round(t["WC"], 3),
        "rho_LC": np.round(t["LC"], 3),
    })


if __name__ == "__main__":
    main()

Generating cohort for kappa = 0.00...
  Saved to: ./outputs/cohort_kappa0.00.csv
Generating cohort for kappa = 0.25...
  Saved to: ./outputs/cohort_kappa0.25.csv
Generating cohort for kappa = 0.50...
  Saved to: ./outputs/cohort_kappa0.50.csv
Generating cohort for kappa = 0.75...
  Saved to: ./outputs/cohort_kappa0.75.csv
Generating cohort for kappa = 1.00...
  Saved to: ./outputs/cohort_kappa1.00.csv
  Saved Excel to: ./outputs/cohort_kappa1.00.xlsx

Validation report saved to: ./outputs/validation_report.txt

Synthetic pediatric growth cohort -- validation report
seed=20260805  N=1020  visits=12
visit ages (months): [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 30.0, 36.0, 48.0, 60.0]

Correlation structure follows Scheffler, Greil & Hermanussen (2017):
head circumference decouples from weight/length after ~24 months.
MAGNITUDES ARE PLACEHOLDERS pending verified literature values.

--- kappa = 0.00 ---
Between-variable correlation of TRUE z-scores (achieved vs target):
   age          

In [5]:
#!/usr/bin/env python3
"""
====================================================================
Synthetic pediatric growth cohort generator (males, 0-60 months)
====================================================================

Generates longitudinal weight / length / head-circumference trajectories
with an EXPLICIT, LITERATURE-GROUNDED, AGE-DEPENDENT correlation structure,
for use in the coupled Neural ODE experiments.

Design rationale
----------------
Two dependence structures are imposed SEPARATELY, because the experiments
must be able to distinguish them:

  1. WITHIN-variable tracking (across age).  A child near the 20th weight
     percentile at 6 months tends to stay near it at 12 months.  This is
     what makes FORECASTING possible.  Implemented as a permanent child
     effect plus an Ornstein-Uhlenbeck (mean-reverting) component, so
     tracking decays with age gap toward a floor rather than being
     deterministic.

  2. BETWEEN-variable coupling (at a given age), R(t).  This is what makes
     CROSS-VARIABLE INFERENCE possible.  Because an unobserved variable in
     that task is never measured at any visit, tracking cannot substitute
     for it -- so cross-variable results measure coupling alone.

R(t) is AGE-DEPENDENT and PAIR-SPECIFIC.  Following the affirmative
(Geraedts et al., 2011) position adopted in the paper, the head-circumference
correlations decline from 0.52 at birth to a 0.32 plateau across a logistic
transition centred at 12 months (width 5 months), essentially complete by
24 months, while the weight-length correlation is held near-constant
(0.58 -> 0.52).  Realised values reproduce Table 1 of the paper.

*** THE COUPLING IS IMPOSED, NOT DISCOVERED. ***
Synthetic data cannot establish that growth variables are biologically
coupled.  It can establish whether a model LEARNS a coupling that is
present.  All claims must be confined to the estimator, not the biology.
The KAPPA sweep (see below) exists so that conclusions do not depend on
committing to one contested correlation value.

Outputs
-------
  cohort_kappa1.00.xlsx   main cohort (kappa = 1)
  cohort_kappa*.csv       sensitivity sweep incl. kappa = 0 negative control
  validation_report.txt   achieved vs target correlations, zone composition

Reproducibility: fully determined by SEED and the CONFIG block below.
The kappa = 1 cohort produced here is byte-for-byte identical to the
released ``pediatric_cohort_coupled.csv``.

Usage:  python generate_cohort.py [--outdir DIR]
"""

import argparse
import os
import sys

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.stats import norm

# =====================================================================
# CONFIG  -- everything reproducible is set here
# =====================================================================

SEED = 20260805
N_CHILDREN = 1020
SEX = 1  # males only

# Clinical visit schedule (months).
VISIT_AGES = [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 30.0, 36.0, 48.0, 60.0]
HISTORICAL_CUTOFF = 24.0

# ---------------------------------------------------------------------
# Target between-variable correlations of z-scores.
# Paper-anchored values (Geraedts affirmative position; reproduce Table 1).
# Weight-length: moderate, near-stable.  Head-circumference pairs: decline
# from 0.52 at birth to a 0.32 plateau across a transition centred at 12 mo.
# ---------------------------------------------------------------------
RHO_WL_EARLY, RHO_WL_LATE = 0.58, 0.52   # weight-length
RHO_WC_EARLY, RHO_WC_LATE = 0.52, 0.32   # weight-headcirc
RHO_LC_EARLY, RHO_LC_LATE = 0.52, 0.32   # length-headcirc

TRANSITION_CENTER = 12.0  # months; logistic centre of the HC decline
TRANSITION_WIDTH = 5.0    # months; smoothness of the logistic transition

# ---------------------------------------------------------------------
# Within-variable tracking.
#   z(t) = sqrt(lambda)*u  +  sqrt(1-lambda)*OU(t)
# lambda = permanent share (sets the long-run tracking floor)
# tau    = OU correlation time in months
# ---------------------------------------------------------------------
LAMBDA_PERM = {"W": 0.45, "L": 0.55, "C": 0.50}
TAU_OU = {"W": 30.0, "L": 40.0, "C": 36.0}

# ---------------------------------------------------------------------
# Measurement noise (physical units), applied to OBSERVED records only.
# ---------------------------------------------------------------------
NOISE_SD = {"W": 0.05, "L": 0.30, "C": 0.20}  # kg, cm, cm

# Missing-visit rate (fraction of ELIGIBLE historical visits deleted at
# random). Birth visit and forecast-horizon visits are never deleted.
# 0.08 over the 7 eligible visits -> ~4.66% of all child-visits.
MISSING_RATE = 0.08

# Coupling-strength sweep for Experiment 6. kappa scales the OFF-DIAGONAL
# entries of R(t). kappa = 0 is the negative control; kappa = 1 is the
# literature-anchored structure.
KAPPA_SWEEP = [0.0, 0.25, 0.5, 0.75, 1.0]
KAPPA_MAIN = 1.0

# ---------------------------------------------------------------------
# Designed at-risk cases (paper Section 5.9, Table 2: "6 designated
# at-risk cases, IDs 201-206").  Their latent z-trajectories are HARD-SET
# after the coupled draw, so they are fixed illustrative children carried
# unchanged through every experiment and every kappa.  Injection happens in
# z-space, so risk zones follow from the same LMS reference the classifier
# uses (internally consistent), and because no random numbers are drawn,
# every OTHER child remains byte-for-byte identical to the un-injected
# cohort.  Values below are z-scores aligned to VISIT_AGES (12 entries).
#
# ID 201 is specified by the paper (Sec. 5.9 / 6.1.3 / Table 5):
# progressive stunting -- weight and length decline from within the
# conditioning window while head circumference is preserved; already
# Warning at 24 mo (z_w=-1.50, z_l=-1.70), length crosses p3 at 30 mo and
# weight at 36 mo, so overall Critical at every forecast horizon; head
# circumference Healthy throughout (z_c=-1.00 at 60 mo).  The 60-mo weight
# and length reproduce Table 5 (13.89 kg, 97.92 cm); Table 5's head-circ
# 49.24 cm is inconsistent with the text's Healthy/z_c=-1.00 and is not
# reproduced -- z_c=-1.00 gives 51.23 cm, Healthy.
#
# IDs 202-206 are NOT individually specified by the paper.  They are
# constructed at-risk archetypes so that the "6 designated cases" claim
# holds; edit freely.
# ---------------------------------------------------------------------
INJECT_DESIGNED_CASES = False

DESIGNED_CASES = {
    # 201: progressive stunting (paper's illustrative case)
    201: {
        "W": [-0.15, -0.20, -0.28, -0.42, -0.60, -0.80, -1.10, -1.50, -1.72, -2.00, -2.45, -2.89],
        "L": [-0.20, -0.28, -0.40, -0.58, -0.78, -0.95, -1.30, -1.70, -1.95, -2.10, -2.40, -2.56],
        "C": [ 0.10,  0.05,  0.00, -0.10, -0.20, -0.30, -0.45, -0.60, -0.70, -0.80, -0.92, -1.00],
    },
    # 202: wasting (acute low weight; length and head circ near-normal)
    202: {
        "W": [-0.30, -0.50, -0.80, -1.20, -1.60, -1.90, -2.10, -2.20, -2.25, -2.30, -2.30, -2.30],
        "L": [-0.10, -0.15, -0.20, -0.25, -0.30, -0.35, -0.40, -0.45, -0.50, -0.55, -0.60, -0.65],
        "C": [ 0.00, -0.05, -0.10, -0.10, -0.15, -0.15, -0.20, -0.20, -0.25, -0.25, -0.30, -0.30],
    },
    # 203: chronic stunting (length-led; weight follows mildly; head preserved)
    203: {
        "W": [-0.40, -0.50, -0.60, -0.75, -0.85, -0.95, -1.05, -1.15, -1.25, -1.30, -1.35, -1.40],
        "L": [-0.50, -0.70, -1.00, -1.40, -1.75, -2.00, -2.20, -2.35, -2.45, -2.50, -2.55, -2.60],
        "C": [ 0.00,  0.00, -0.05, -0.10, -0.10, -0.15, -0.15, -0.20, -0.20, -0.25, -0.25, -0.30],
    },
    # 204: overweight / high-side (weight crosses p97 upward)
    204: {
        "W": [ 0.30,  0.50,  0.80,  1.10,  1.35,  1.55,  1.75,  1.90,  2.00,  2.05,  2.10,  2.15],
        "L": [ 0.20,  0.30,  0.40,  0.50,  0.60,  0.65,  0.70,  0.75,  0.80,  0.85,  0.90,  0.95],
        "C": [ 0.10,  0.15,  0.20,  0.25,  0.30,  0.35,  0.40,  0.45,  0.50,  0.55,  0.60,  0.65],
    },
    # 205: head-circumference concern (low/declining head circ; W and L normal)
    205: {
        "W": [-0.20, -0.25, -0.30, -0.35, -0.40, -0.45, -0.50, -0.55, -0.60, -0.60, -0.65, -0.65],
        "L": [-0.15, -0.20, -0.25, -0.30, -0.35, -0.40, -0.45, -0.50, -0.55, -0.55, -0.60, -0.60],
        "C": [-0.30, -0.50, -0.80, -1.20, -1.55, -1.80, -2.00, -2.15, -2.25, -2.30, -2.35, -2.40],
    },
    # 206: late-onset mixed (Healthy through 24 mo; deteriorates only over
    #      the forecast horizon -- a pure forecast-detection test)
    206: {
        "W": [-0.10, -0.10, -0.15, -0.20, -0.25, -0.30, -0.40, -0.55, -0.90, -1.30, -1.90, -2.30],
        "L": [-0.15, -0.15, -0.20, -0.25, -0.30, -0.35, -0.45, -0.60, -0.95, -1.35, -1.95, -2.35],
        "C": [ 0.00,  0.00, -0.05, -0.05, -0.10, -0.10, -0.15, -0.25, -0.40, -0.55, -0.75, -0.90],
    },
}

# ---------------------------------------------------------------------
# Reference distribution: LMS-style parameters per variable per age.
# M (median) and SD (physical units); L is the Box-Cox power (L=1 normal,
# L<1 right skew, used for weight).  Classifier uses THESE SAME parameters,
# so cohort and classifier are internally consistent by construction.
# ---------------------------------------------------------------------
REF_AGES = [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 36.0, 48.0, 60.0]

REF_M = {
    "W": [3.490, 5.780, 6.680, 7.655, 8.575, 9.390, 10.570, 11.500, 12.980, 14.695, 16.440],
    "L": [50.915, 58.375, 62.915, 68.440, 72.940, 76.350, 81.885, 85.940, 92.350, 99.120, 105.375],
    "C": [34.920, 39.430, 41.930, 44.445, 45.940, 47.420, 48.950, 49.900, 51.385, 52.225, 52.790],
}
REF_SD = {
    "W": [0.491, 0.708, 0.702, 0.689, 0.628, 0.617, 0.629, 0.618, 0.624, 0.790, 0.984],
    "L": [2.279, 2.301, 2.295, 2.292, 2.286, 2.290, 2.298, 2.324, 2.320, 2.615, 2.907],
    "C": [1.522, 1.514, 1.521, 1.521, 1.529, 1.525, 1.518, 1.514, 1.519, 1.528, 1.565],
}
REF_L = {"W": -0.30, "L": 1.0, "C": 1.0}

# Risk-zone band boundaries (percentiles). Matches the paper's 15/85
# Healthy convention; change here to switch to 10/90.
P_CRIT_LO, P_WARN_LO, P_WARN_HI, P_CRIT_HI = 3, 15, 85, 97

VARS = ["W", "L", "C"]
VARNAME = {"W": "Weight_kg", "L": "Length_cm", "C": "HeadCircumference_cm"}


# =====================================================================
# Correlation structure
# =====================================================================

def _transition(ages):
    """Logistic 0->1 ramp centred at TRANSITION_CENTER (0 = early, 1 = late)."""
    return 1.0 / (1.0 + np.exp(-(np.asarray(ages, float) - TRANSITION_CENTER) / TRANSITION_WIDTH))


def target_correlations(ages):
    """Age-dependent pair correlations. Returns dict of arrays over ages."""
    s = _transition(ages)
    return {
        "WL": RHO_WL_EARLY + (RHO_WL_LATE - RHO_WL_EARLY) * s,
        "WC": RHO_WC_EARLY + (RHO_WC_LATE - RHO_WC_EARLY) * s,
        "LC": RHO_LC_EARLY + (RHO_LC_LATE - RHO_LC_EARLY) * s,
    }


def correlation_matrix(rho_wl, rho_wc, rho_lc, kappa=1.0):
    """3x3 correlation matrix with off-diagonals scaled by kappa."""
    R = np.array([
        [1.0, kappa * rho_wl, kappa * rho_wc],
        [kappa * rho_wl, 1.0, kappa * rho_lc],
        [kappa * rho_wc, kappa * rho_lc, 1.0],
    ])
    return _nearest_pd(R)


def _nearest_pd(R):
    """Project to nearest positive-definite correlation matrix (eigenvalue clip)."""
    w, V = np.linalg.eigh(R)
    if w.min() > 1e-8:
        return R
    w = np.clip(w, 1e-8, None)
    R = V @ np.diag(w) @ V.T
    d = np.sqrt(np.diag(R))
    return R / np.outer(d, d)


# =====================================================================
# Latent z-score generation
# =====================================================================

def generate_latent_z(rng, n, ages, kappa):
    """
    Latent standardized growth positions, shape (n, n_ages, 3).

    Step 1: independent per-variable paths with tracking
            z_v(t) = sqrt(lam)*u_v + sqrt(1-lam)*OU_v(t)
    Step 2: at EACH age, mix across variables by the Cholesky factor of
            R(t). Since each variable's path is marginally N(0,1), the
            mixture is marginally N(0,1) and has cross-sectional
            correlation exactly R(t), while tracking is preserved because
            all three variables share the same temporal structure.
    """
    ages = np.asarray(ages, float)
    n_ages = len(ages)
    indep = np.empty((n, n_ages, 3))

    for k, v in enumerate(VARS):
        lam = LAMBDA_PERM[v]
        tau = TAU_OU[v]
        u = rng.standard_normal(n)                      # permanent child effect
        ou = np.empty((n, n_ages))
        ou[:, 0] = rng.standard_normal(n)               # stationary start
        for i in range(1, n_ages):
            dt = ages[i] - ages[i - 1]
            phi = np.exp(-dt / tau)                     # OU autocorrelation
            ou[:, i] = phi * ou[:, i - 1] + np.sqrt(1 - phi**2) * rng.standard_normal(n)
        indep[:, :, k] = np.sqrt(lam) * u[:, None] + np.sqrt(1 - lam) * ou

    rho = target_correlations(ages)
    z = np.empty_like(indep)
    for i in range(n_ages):
        R = correlation_matrix(rho["WL"][i], rho["WC"][i], rho["LC"][i], kappa)
        Lc = np.linalg.cholesky(R)
        z[:, i, :] = indep[:, i, :] @ Lc.T
    return z


def inject_designed_cases(z, ages):
    """Overwrite the latent z-trajectories of the designated at-risk children
    (IDs in DESIGNED_CASES) in place.  z has shape (N, n_ages, 3) with children
    indexed 0..N-1 (Child_ID = index + 1) and the last axis ordered as VARS
    = [W, L, C].  Draws no randomness, so all non-designed children are left
    untouched.  Returns the list of IDs actually injected."""
    var_idx = {v: k for k, v in enumerate(VARS)}
    n_ages = len(ages)
    injected = []
    for cid, spec in DESIGNED_CASES.items():
        row = cid - 1
        if row < 0 or row >= z.shape[0]:
            continue                                   # ID outside cohort; skip
        for v, series in spec.items():
            if len(series) != n_ages:
                raise ValueError(
                    f"Designed case {cid}, variable {v}: expected {n_ages} "
                    f"z-values (one per VISIT_AGES), got {len(series)}.")
            z[row, :, var_idx[v]] = np.asarray(series, float)
        injected.append(cid)
    return injected


# =====================================================================
# z-score  <->  physical units  (LMS)
# =====================================================================

def _ref_interp(var, ages):
    """Interpolate reference M and SD to arbitrary ages (monotone PCHIP)."""
    M = PchipInterpolator(REF_AGES, REF_M[var])(ages)
    SD = PchipInterpolator(REF_AGES, REF_SD[var])(ages)
    return M, SD


def z_to_value(var, z, ages):
    """LMS inverse: z-score -> physical measurement."""
    M, SD = _ref_interp(var, ages)
    S = SD / M                      # coefficient of variation
    Lp = REF_L[var]
    if abs(Lp - 1.0) < 1e-12:
        return M + SD * z
    return M * np.power(np.clip(1.0 + Lp * S * z, 1e-6, None), 1.0 / Lp)


def value_to_z(var, x, ages):
    """LMS forward: physical measurement -> z-score."""
    M, SD = _ref_interp(var, ages)
    S = SD / M
    Lp = REF_L[var]
    if abs(Lp - 1.0) < 1e-12:
        return (x - M) / SD
    return (np.power(x / M, Lp) - 1.0) / (Lp * S)


def classify(z):
    """Risk zone from z-score, using the band boundaries in CONFIG."""
    z_crit_lo, z_warn_lo = norm.ppf(P_CRIT_LO / 100), norm.ppf(P_WARN_LO / 100)
    z_warn_hi, z_crit_hi = norm.ppf(P_WARN_HI / 100), norm.ppf(P_CRIT_HI / 100)
    out = np.full(np.shape(z), "Healthy", dtype=object)
    out[(z >= z_crit_lo) & (z < z_warn_lo)] = "Warning"
    out[(z > z_warn_hi) & (z <= z_crit_hi)] = "Warning"
    out[(z < z_crit_lo) | (z > z_crit_hi)] = "Critical"
    return out


SEVERITY = {"Healthy": 0, "Warning": 1, "Critical": 2}


# =====================================================================
# Cohort assembly
# =====================================================================

def build_cohort(kappa, seed=SEED):
    """Build one cohort at a given coupling strength kappa."""
    rng = np.random.default_rng(seed)
    ages = np.array(VISIT_AGES, float)
    n_ages = len(ages)

    z = generate_latent_z(rng, N_CHILDREN, ages, kappa)

    if INJECT_DESIGNED_CASES:
        inject_designed_cases(z, ages)                 # overwrite IDs 201-206

    rows = {
        "Child_ID": np.repeat(np.arange(1, N_CHILDREN + 1), n_ages),
        "Sex": SEX,
        "Age_Months": np.tile(ages, N_CHILDREN),
    }

    age_flat = rows["Age_Months"]
    for k, v in enumerate(VARS):
        z_flat = z[:, :, k].ravel()
        true_val = z_to_value(v, z_flat, age_flat)

        # Observed = true + independent measurement noise
        obs_val = true_val + rng.normal(0.0, NOISE_SD[v], size=true_val.shape)
        obs_z = value_to_z(v, obs_val, age_flat)

        rows[f"{VARNAME[v]}_true"] = np.round(true_val, 3)
        rows[VARNAME[v]] = np.round(obs_val, 2)
        rows[f"z_{v}"] = np.round(z_flat, 4)             # ground-truth z
        rows[f"z_{v}_obs"] = np.round(obs_z, 4)
        rows[f"zone_{v}"] = classify(z_flat)

    df = pd.DataFrame(rows)

    # Overall status = most severe zone across the three variables
    sev = np.stack([df[f"zone_{v}"].map(SEVERITY).values for v in VARS])
    inv = {v: k for k, v in SEVERITY.items()}
    df["zone_overall"] = [inv[s] for s in sev.max(axis=0)]

    # Missingness: emulate missed appointments. Never delete the birth visit
    # (needed as an anchor) nor forecast-horizon visits (needed as ground truth).
    eligible = (df.Age_Months > 0) & (df.Age_Months <= HISTORICAL_CUTOFF)
    df["Observed"] = True
    drop = eligible & (rng.random(len(df)) < MISSING_RATE)
    df.loc[drop, "Observed"] = False
    for v in VARS:
        df.loc[drop, VARNAME[v]] = np.nan
        df.loc[drop, f"z_{v}_obs"] = np.nan

    df["Phase"] = np.where(df.Age_Months <= HISTORICAL_CUTOFF, "Historical", "Forecast")
    df["kappa"] = kappa
    return df


# =====================================================================
# Validation
# =====================================================================

def validate(df, kappa):
    """Check achieved correlations and tracking against targets."""
    lines = []
    ages = np.array(VISIT_AGES, float)
    tgt = target_correlations(ages)

    lines.append(f"--- kappa = {kappa:.2f} ---")
    lines.append("Between-variable correlation of TRUE z-scores (achieved vs target):")
    lines.append(f"{'age':>6} {'r(W,L)':>16} {'r(W,C)':>16} {'r(L,C)':>16}")
    for i, a in enumerate(ages):
        g = df[df.Age_Months == a]
        rwl = g.z_W.corr(g.z_L)
        rwc = g.z_W.corr(g.z_C)
        rlc = g.z_L.corr(g.z_C)
        lines.append(
            f"{a:6.1f} {rwl:7.3f} /{kappa*tgt['WL'][i]:6.3f} "
            f"{rwc:7.3f} /{kappa*tgt['WC'][i]:6.3f} "
            f"{rlc:7.3f} /{kappa*tgt['LC'][i]:6.3f}"
        )

    lines.append("")
    lines.append("Within-variable tracking, corr(z at birth, z at age):")
    for v in VARS:
        p = df.pivot(index="Child_ID", columns="Age_Months", values=f"z_{v}")
        lines.append(f"  {v}: " + " ".join(f"{p[a].corr(p[0.0]):.2f}" for a in ages))

    lines.append("")
    lines.append("Risk-zone composition (all child-visits):")
    for v in VARS:
        vc = df[f"zone_{v}"].value_counts(normalize=True)
        lines.append(
            f"  {v}: H={vc.get('Healthy',0):.1%}  "
            f"W={vc.get('Warning',0):.1%}  C={vc.get('Critical',0):.1%}"
        )
    vc = df.zone_overall.value_counts(normalize=True)
    lines.append(
        f"  overall: H={vc.get('Healthy',0):.1%}  "
        f"W={vc.get('Warning',0):.1%}  C={vc.get('Critical',0):.1%}"
    )
    lines.append("")
    lines.append(f"Missing observed records: {(~df.Observed).mean():.1%}")
    lines.append("")
    return "\n".join(lines)


# =====================================================================
# Main
# =====================================================================

def validate_designed(df):
    """Report the risk-zone trajectory of each designed at-risk child."""
    if not INJECT_DESIGNED_CASES or not DESIGNED_CASES:
        return "Designed-case injection: OFF."
    lines = ["Designed at-risk cases (overall zone by age):"]
    ages = VISIT_AGES
    lines.append("   ID  " + " ".join(f"{a:>5g}" for a in ages))
    zabbr = {"Healthy": "H", "Warning": "W", "Critical": "C"}
    for cid in sorted(DESIGNED_CASES):
        g = df[df.Child_ID == cid].set_index("Age_Months")
        if len(g) == 0:
            continue
        row = " ".join(f"{zabbr[g.loc[a, 'zone_overall']]:>5}" for a in ages)
        lines.append(f"  {cid:>3}  {row}")
    # spell out 201 against the paper's stated anchors
    g201 = df[df.Child_ID == 201].set_index("Age_Months")
    if len(g201):
        lines.append("")
        lines.append("  201 check vs paper: "
                     f"z_w(24)={g201.loc[24.0,'z_W']:.2f} (paper -1.50), "
                     f"z_l(24)={g201.loc[24.0,'z_L']:.2f} (paper -1.70)")
        lines.append("            60 mo true: "
                     f"W={g201.loc[60.0,'Weight_kg_true']:.2f} kg (Table 5 13.89), "
                     f"L={g201.loc[60.0,'Length_cm_true']:.2f} cm (Table 5 97.92), "
                     f"C={g201.loc[60.0,'HeadCircumference_cm_true']:.2f} cm")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--outdir", default="./outputs")
    # Parse known args only, ignore extras like -f from Jupyter/Colab
    args, unknown = ap.parse_known_args()

    if unknown:
        print(f"Warning: Ignoring unknown arguments: {unknown}", file=sys.stderr)

    os.makedirs(args.outdir, exist_ok=True)

    report = [
        "Synthetic pediatric growth cohort -- validation report",
        "=" * 60,
        f"seed={SEED}  N={N_CHILDREN}  visits={len(VISIT_AGES)}",
        f"visit ages (months): {VISIT_AGES}",
        "",
        "Correlation structure (Geraedts affirmative position):",
        "head circumference declines 0.52 -> 0.32 across a transition centred",
        "at 12 months (width 5); weight-length held near-constant 0.58 -> 0.52.",
        "Realised values reproduce Table 1 of the paper.",
        "",
    ]

    for kappa in KAPPA_SWEEP:
        print(f"Generating cohort for kappa = {kappa:.2f}...")
        df = build_cohort(kappa)
        report.append(validate(df, kappa))
        path = os.path.join(args.outdir, f"cohort_kappa{kappa:.2f}.csv")
        df.to_csv(path, index=False)
        print(f"  Saved to: {path}")

        if abs(kappa - KAPPA_MAIN) < 1e-12:
            report.append(validate_designed(df))
            xls = os.path.join(args.outdir, f"cohort_kappa{kappa:.2f}.xlsx")
            with pd.ExcelWriter(xls, engine="openpyxl") as w:
                df.to_excel(w, sheet_name="cohort", index=False)
                _readme_sheet().to_excel(w, sheet_name="README", index=False)
                _corr_sheet().to_excel(w, sheet_name="R_of_t", index=False)
            print(f"  Saved Excel to: {xls}")

    rpt = os.path.join(args.outdir, "validation_report.txt")
    with open(rpt, "w") as f:
        f.write("\n".join(report))
    print(f"\nValidation report saved to: {rpt}")
    print("\n" + "\n".join(report))


def _readme_sheet():
    return pd.DataFrame({
        "Field": [
            "Child_ID", "Sex", "Age_Months", "Phase", "Observed",
            "Weight_kg / Length_cm / HeadCircumference_cm",
            "*_true", "z_W / z_L / z_C", "z_*_obs",
            "zone_W / zone_L / zone_C", "zone_overall", "kappa",
        ],
        "Description": [
            "Child identifier (1..N)",
            "1 = male (cohort is males only)",
            "Age at visit in months",
            "Historical (<=24mo, conditioning) or Forecast (>24mo)",
            "FALSE = visit missed; measurements are blank",
            "OBSERVED measurement = true value + measurement noise",
            "Noiseless ground-truth value (evaluation only; not model input)",
            "Ground-truth z-score against the reference distribution",
            "z-score of the noisy observed value",
            "Per-variable risk zone from ground-truth z",
            "Most severe zone across the three variables",
            "Coupling-strength scaling of R(t) off-diagonals",
        ],
    })


def _corr_sheet():
    ages = np.array(VISIT_AGES, float)
    t = target_correlations(ages)
    return pd.DataFrame({
        "Age_Months": ages,
        "rho_WL": np.round(t["WL"], 3),
        "rho_WC": np.round(t["WC"], 3),
        "rho_LC": np.round(t["LC"], 3),
    })


if __name__ == "__main__":
    main()

Generating cohort for kappa = 0.00...
  Saved to: ./outputs/cohort_kappa0.00.csv
Generating cohort for kappa = 0.25...
  Saved to: ./outputs/cohort_kappa0.25.csv
Generating cohort for kappa = 0.50...
  Saved to: ./outputs/cohort_kappa0.50.csv
Generating cohort for kappa = 0.75...
  Saved to: ./outputs/cohort_kappa0.75.csv
Generating cohort for kappa = 1.00...
  Saved to: ./outputs/cohort_kappa1.00.csv
  Saved Excel to: ./outputs/cohort_kappa1.00.xlsx

Validation report saved to: ./outputs/validation_report.txt

Synthetic pediatric growth cohort -- validation report
seed=20260805  N=1020  visits=12
visit ages (months): [0.0, 1.5, 3.0, 6.0, 9.0, 12.0, 18.0, 24.0, 30.0, 36.0, 48.0, 60.0]

Correlation structure (Geraedts affirmative position):
head circumference declines 0.52 -> 0.32 across a transition centred
at 12 months (width 5); weight-length held near-constant 0.58 -> 0.52.
Realised values reproduce Table 1 of the paper.

--- kappa = 0.00 ---
Between-variable correlation of TRUE z-s